# Encoder-Decoder Architecture
### Two towers, three kinds of attention — the full Transformer assembled

---

## 📖 The Story

By mid-2017 the team behind *Attention Is All You Need* had every piece: self-attention (Topic 2), multi-head attention (Topic 3), and positional encoding (Topic 4). One question was left — *how do you wire these into a model that reads one sequence and writes another?*

Their answer was a **two-tower** design: an **encoder** that reads the whole source at once, and a **decoder** that generates the target one token at a time while consulting both its own past output and the encoder. This notebook assembles that exact machine from scratch — encoder layers, decoder layers, Add & Norm, the feed-forward network, and all **three kinds of attention** wired together.

---

**What this notebook covers:**
- Building an encoder layer (self-attention + FFN, each wrapped in Add & Norm)
- Building a decoder layer (masked self-attention → cross-attention → FFN)
- Wiring the encoder output into the decoder via cross-attention
- Visualizing all three attention maps and the causal mask
- Verifying the masked self-attention against PyTorch, and the full stack against `nn.Transformer`

**Prerequisites:**
- Self-Attention (Topic 2), Multi-Head Attention (Topic 3), Positional Encoding (Topic 4)
- NumPy fundamentals and matrix multiplication

**Note:** Like earlier topics, we use a small in-notebook example so every matrix stays inspectable. Weights are random (simulating a trained state) — the goal is to understand the *wiring*, not to train a translator.

In [ ]:
# --- All Imports ---
import numpy as np                        # Core math — encoder-decoder from scratch
import matplotlib.pyplot as plt           # Plotting
import seaborn as sns                     # Heatmap visualizations
import torch                              # PyTorch — verification
import torch.nn as nn                     # nn.Transformer
import torch.nn.functional as F           # Scaled dot-product attention
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)                        # Reproducibility
torch.manual_seed(42)

print('All imports successful ✅')

## Part 1: Theory Recap

Five things to know before we write any code:

- **The encoder is the reader, the decoder is the writer**: The encoder turns the source into context-rich representations; the decoder generates the target token by token, conditioned on the encoder.
- **The encoder layer has two sub-layers**: multi-head **self-attention**, then a position-wise **feed-forward network (FFN)** — each wrapped in *Add & Norm* (residual + LayerNorm).
- **The decoder layer has three sub-layers**: **masked self-attention** (can't peek ahead) → **cross-attention** (looks at the encoder) → **FFN** — each again wrapped in *Add & Norm*.
- **Three kinds of attention appear**: encoder self-attention, decoder masked self-attention, and encoder–decoder **cross-attention** (the bridge, where Query comes from the decoder and Key/Value from the encoder — exactly Topic 1's cross-attention).
- **Add & Norm makes depth trainable**: residual connections let gradients skip past sub-layers and LayerNorm keeps activations stable, so we can stack many layers.

## Setting Up a Real Example Sentence

We translate **"The cat sat on mat"** (source, 5 tokens) into a short target **"&lt;s&gt; Le chat assis"** (4 tokens, starting with a start token). Source and target have **different lengths** — exactly the situation an encoder–decoder is built for.

Each token gets a small random embedding. In a real model these come from learned embedding layers plus positional encoding (Topic 4); here we keep them small so every matrix is readable.

In [ ]:
# Source and target sequences (different lengths — the whole point of seq2seq)
src_words = ['The', 'cat', 'sat', 'on', 'mat']    # encoder input
tgt_words = ['<s>', 'Le', 'chat', 'assis']        # decoder input (shifted-right target)

L_src, L_tgt = len(src_words), len(tgt_words)
d_model = 16    # embedding dimension
n_heads = 4     # attention heads  ->  d_k = d_model // n_heads = 4
d_ff    = 32    # hidden size of the feed-forward network

# Input embeddings (in a real model: token embedding + positional encoding)
X_src = np.random.randn(L_src, d_model) * 0.3   # encoder input  (5, 16)
Y_tgt = np.random.randn(L_tgt, d_model) * 0.3   # decoder input  (4, 16)

print(f'Source : {src_words}  -> X_src shape {X_src.shape}')
print(f'Target : {tgt_words}  -> Y_tgt shape {Y_tgt.shape}')
print(f'\nd_model={d_model}, n_heads={n_heads}, d_k={d_model//n_heads}, d_ff={d_ff}')

## Part 2: Building the Encoder-Decoder From Scratch

We build four reusable pieces, then wire them together:
1. `softmax`, `layer_norm`, `ffn` — the small building blocks
2. `multi_head_attention` — reused from Topic 3, now with an optional **mask**
3. `encoder_layer` — self-attention + FFN, each with Add & Norm
4. `decoder_layer` — masked self-attention → cross-attention → FFN, each with Add & Norm

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax (subtract max before exp)."""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def layer_norm(x, eps=1e-5):
    """LayerNorm: normalize each token vector to mean 0, variance 1."""
    mu  = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def ffn(x, W1, b1, W2, b2):
    """Position-wise feed-forward network: ReLU(xW1+b1)W2+b2."""
    return np.maximum(0, x @ W1 + b1) @ W2 + b2

def multi_head_attention(Q_in, K_in, V_in, WQ, WK, WV, WO, n_heads, mask=None):
    """Multi-head attention (Topic 3) with an OPTIONAL additive mask.
    Q_in supplies the queries; K_in/V_in supply keys/values.
    For self-attention they are identical; for CROSS-attention K_in/V_in = encoder output."""
    L_q, d_model = Q_in.shape
    L_k = K_in.shape[0]
    d_k = d_model // n_heads
    # Project, then split into heads: (L, d_model) -> (n_heads, L, d_k)
    Q = (Q_in @ WQ).reshape(L_q, n_heads, d_k).transpose(1, 0, 2)
    K = (K_in @ WK).reshape(L_k, n_heads, d_k).transpose(1, 0, 2)
    V = (V_in @ WV).reshape(L_k, n_heads, d_k).transpose(1, 0, 2)
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)   # (n_heads, L_q, L_k)
    if mask is not None:
        scores = scores + mask          # INTERVIEW NOTE: mask adds -inf to forbidden positions
    weights = softmax(scores, axis=-1)  # row-wise; each query distributes attention over keys
    context = np.matmul(weights, V)                      # (n_heads, L_q, d_k)
    context = context.transpose(1, 0, 2).reshape(L_q, d_model)   # concat heads back
    return context @ WO, weights

# --- Parameter initialization (random = 'pretend trained') ---
rand = lambda *s: np.random.randn(*s) * 0.3
def attn_params():  # WQ, WK, WV, WO  (all d_model x d_model)
    return tuple(rand(d_model, d_model) for _ in range(4))
def ffn_params():   # W1, b1, W2, b2
    return rand(d_model, d_ff), np.zeros(d_ff), rand(d_ff, d_model), np.zeros(d_model)

print('Building blocks ready: softmax, layer_norm, ffn, multi_head_attention ✅')

In [ ]:
def encoder_layer(x, attn_p, ffn_p, n_heads):
    """Encoder layer: Self-Attention -> Add&Norm -> FFN -> Add&Norm."""
    attn_out, w_self = multi_head_attention(x, x, x, *attn_p, n_heads)
    z   = layer_norm(x + attn_out)        # Add & Norm  #1
    out = layer_norm(z + ffn(z, *ffn_p))  # Add & Norm  #2
    return out, w_self

def decoder_layer(y, enc, self_p, cross_p, ffn_p, n_heads, causal_mask):
    """Decoder layer: Masked Self-Attn -> Cross-Attn -> FFN (each Add&Norm)."""
    # 1) Masked self-attention — a token may only look at itself and earlier tokens
    sa, w_masked = multi_head_attention(y, y, y, *self_p, n_heads, mask=causal_mask)
    a = layer_norm(y + sa)
    # 2) Cross-attention — Query from decoder (a), Key/Value from the ENCODER output
    ca, w_cross = multi_head_attention(a, enc, enc, *cross_p, n_heads)  # the BRIDGE
    b = layer_norm(a + ca)
    # 3) Feed-forward
    out = layer_norm(b + ffn(b, *ffn_p))
    return out, w_masked, w_cross

# Causal mask (L_tgt x L_tgt): 0 on/below diagonal, -inf above (block future tokens)
causal_mask = np.triu(np.full((L_tgt, L_tgt), -1e9), k=1)

# Initialize one encoder layer and one decoder layer
enc_attn, enc_ffn = attn_params(), ffn_params()
dec_self, dec_cross, dec_ffn = attn_params(), attn_params(), ffn_params()

# --- Forward pass ---
print('ENCODER: reading the source...')
enc_out, w_enc_self = encoder_layer(X_src, enc_attn, enc_ffn, n_heads)
print(f'  encoder output (enc) shape: {enc_out.shape}  <- becomes Key/Value for cross-attention')

print('\nDECODER: generating the target...')
dec_out, w_dec_masked, w_dec_cross = decoder_layer(
    Y_tgt, enc_out, dec_self, dec_cross, dec_ffn, n_heads, causal_mask)
print(f'  decoder output (dec) shape: {dec_out.shape}')
print(f'\n  encoder self-attn  weights: {w_enc_self.shape}  (n_heads, L_src, L_src)')
print(f'  decoder masked-self weights: {w_dec_masked.shape}  (n_heads, L_tgt, L_tgt)')
print(f'  cross-attn          weights: {w_dec_cross.shape}  (n_heads, L_tgt, L_src)  <- rectangular!')

## What Just Happened?

We ran one full encoder–decoder forward pass:

- The **encoder** read all 5 source tokens at once and produced `enc` — a context-rich representation of the source.
- The **decoder** took the 4 target tokens and ran **three** sub-layers: masked self-attention, cross-attention, and an FFN.
- **Masked self-attention** produced a square `(4×4)` map — but the upper triangle is forced to zero, so token *i* never sees token *i+1*.
- **Cross-attention** produced a **rectangular** `(4×5)` map — 4 target queries against 5 source keys. This is the bridge: notice it has the *same rectangular shape* as Topic 1's cross-attention heatmap.
- Every sub-layer was wrapped in **Add & Norm**, so the output stays the same shape as the input and gradients could flow through a deep stack.

Stack `N` of each layer and add a final linear+softmax over the vocabulary, and you have the original Transformer.

In [ ]:
# --- Visualisation 1: The Three Kinds of Attention, side by side ---
# We average over heads just for display.
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.heatmap(w_enc_self.mean(0), xticklabels=src_words, yticklabels=src_words,
            annot=True, fmt='.2f', cmap='Blues', ax=axes[0], cbar=False, vmin=0, vmax=1)
axes[0].set_title('1. Encoder Self-Attention\n(source attends to source — square)', fontweight='bold')
axes[0].set_xlabel('Key (source)'); axes[0].set_ylabel('Query (source)')

sns.heatmap(w_dec_masked.mean(0), xticklabels=tgt_words, yticklabels=tgt_words,
            annot=True, fmt='.2f', cmap='Greens', ax=axes[1], cbar=False, vmin=0, vmax=1)
axes[1].set_title('2. Decoder MASKED Self-Attention\n(upper triangle = 0, no peeking ahead)', fontweight='bold')
axes[1].set_xlabel('Key (target)'); axes[1].set_ylabel('Query (target)')

sns.heatmap(w_dec_cross.mean(0), xticklabels=src_words, yticklabels=tgt_words,
            annot=True, fmt='.2f', cmap='Oranges', ax=axes[2], cbar=False, vmin=0, vmax=1)
axes[2].set_title('3. Cross-Attention (THE BRIDGE)\n(target attends to source — rectangular)', fontweight='bold')
axes[2].set_xlabel('Key (source)'); axes[2].set_ylabel('Query (target)')

plt.suptitle('The Three Kinds of Attention in a Transformer', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print('Map 2 is lower-triangular  -> the causal mask is working (no future leakage).')
print('Map 3 is rectangular (4x5) -> target queries reading source keys: the encoder-decoder bridge.')

In [ ]:
# --- Visualisation 2: The Causal Mask + how shapes flow through the stack ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the raw causal mask (what masked self-attention adds before softmax)
mask_display = (causal_mask == 0).astype(int)  # 1 = allowed, 0 = blocked
sns.heatmap(mask_display, xticklabels=tgt_words, yticklabels=tgt_words,
            annot=True, fmt='d', cmap='RdYlGn', ax=axes[0], cbar=False)
axes[0].set_title('Causal Mask (1 = can attend, 0 = blocked)\nlower-triangular by design', fontweight='bold')
axes[0].set_xlabel('Key position'); axes[0].set_ylabel('Query position')

# Right: shape of the data as it flows through the architecture
stages = ['X_src\n(enc in)', 'enc_out\n(context)', 'Y_tgt\n(dec in)',
          'masked\nself-attn', 'cross\n-attn', 'dec_out']
rows    = [L_src, L_src, L_tgt, L_tgt, L_tgt, L_tgt]
colors  = ['#4C72B0','#4C72B0','#55A868','#55A868','#DD8452','#55A868']
axes[1].bar(stages, rows, color=colors)
axes[1].set_title('Sequence length at each stage\n(encoder side = 5, decoder side = 4)', fontweight='bold')
axes[1].set_ylabel('rows (sequence length)')
for i, r in enumerate(rows):
    axes[1].text(i, r + 0.05, str(r), ha='center', fontweight='bold')

plt.tight_layout(); plt.show()
print('Output length follows the DECODER (4), never the encoder (5) — output can differ from input.')

## Part 3: PyTorch Verification

Two checks:
1. **Numeric** — our masked self-attention should match PyTorch's `F.scaled_dot_product_attention(..., is_causal=True)` exactly (this is the core operation, same as Topic 2 but with a causal mask).
2. **End-to-end** — a real `nn.Transformer` should accept our source/target shapes and a causal mask, returning the expected output shape.

In [ ]:
# --- Check 1: numeric match on the decoder's MASKED self-attention (head outputs) ---
WQ, WK, WV, WO = dec_self
d_k = d_model // n_heads
Qh = (Y_tgt @ WQ).reshape(L_tgt, n_heads, d_k).transpose(1, 0, 2)
Kh = (Y_tgt @ WK).reshape(L_tgt, n_heads, d_k).transpose(1, 0, 2)
Vh = (Y_tgt @ WV).reshape(L_tgt, n_heads, d_k).transpose(1, 0, 2)

# Our per-head context (before the WO projection)
our_ctx = np.matmul(w_dec_masked, Vh)   # (n_heads, L_tgt, d_k)

t = lambda z: torch.tensor(z, dtype=torch.float32)
with torch.no_grad():
    ref_ctx = F.scaled_dot_product_attention(t(Qh), t(Kh), t(Vh), is_causal=True).numpy()

max_diff = np.max(np.abs(our_ctx - ref_ctx))
print('=== Check 1: Masked Self-Attention vs PyTorch causal SDPA ===')
print(f'  Max absolute difference: {max_diff:.2e}')
print('  ✅ Match' if max_diff < 1e-4 else '  ❌ Mismatch')

# --- Check 2: a real nn.Transformer accepts our shapes ---
print('\n=== Check 2: torch.nn.Transformer end-to-end ===')
model = nn.Transformer(d_model=d_model, nhead=n_heads, num_encoder_layers=2,
                       num_decoder_layers=2, dim_feedforward=d_ff, batch_first=True)
src_t = torch.randn(1, L_src, d_model)
tgt_t = torch.randn(1, L_tgt, d_model)
tgt_causal = nn.Transformer.generate_square_subsequent_mask(L_tgt)
with torch.no_grad():
    out_t = model(src_t, tgt_t, tgt_mask=tgt_causal)
print(f'  src {tuple(src_t.shape)} + tgt {tuple(tgt_t.shape)}  ->  output {tuple(out_t.shape)}')
print('  ✅ Output length follows the target (decoder), exactly as in our scratch version.')

## Part 4: Hyperparameter Experiments

Two experiments for intuition about real model sizes:
1. **Depth (N layers) vs parameter count** — how fast does a Transformer grow as we stack layers?
2. **Encoder-decoder vs single-tower** — at equal depth, the two-tower model carries more parameters (the price of cross-attention).

In [ ]:
# Approx parameters per layer (attention: 4 x d_model^2; FFN: 2 x d_model x d_ff)
def attn_params_count(dm):       return 4 * dm * dm
def ffn_params_count(dm, dff):   return 2 * dm * dff

def enc_layer_params(dm, dff):   return attn_params_count(dm) + ffn_params_count(dm, dff)        # self + ffn
def dec_layer_params(dm, dff):   return 2 * attn_params_count(dm) + ffn_params_count(dm, dff)    # self + cross + ffn

dm, dff = 512, 2048   # original Transformer base sizes
N_values = [1, 2, 4, 6, 8, 12]

enc_dec_total = [N * (enc_layer_params(dm, dff) + dec_layer_params(dm, dff)) for N in N_values]
enc_only_total = [N * enc_layer_params(dm, dff) for N in N_values]   # BERT-style single tower

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(N_values, np.array(enc_dec_total)/1e6, marker='o', linewidth=2, color='steelblue')
axes[0].set_title('Encoder-Decoder size vs depth N\n(d_model=512, d_ff=2048)', fontweight='bold')
axes[0].set_xlabel('Layers per stack (N)'); axes[0].set_ylabel('Parameters (millions)')
axes[0].grid(True, alpha=0.3)
for x, y in zip(N_values, enc_dec_total):
    axes[0].annotate(f'{y/1e6:.0f}M', (x, y/1e6), textcoords='offset points', xytext=(0,8), ha='center', fontsize=8)

axes[1].bar([n-0.2 for n in N_values], np.array(enc_only_total)/1e6, width=0.4, label='Encoder-only (BERT-like)', color='#55A868')
axes[1].bar([n+0.2 for n in N_values], np.array(enc_dec_total)/1e6, width=0.4, label='Encoder-Decoder', color='#DD8452')
axes[1].set_title('Two towers cost more than one\n(same depth, same dimensions)', fontweight='bold')
axes[1].set_xlabel('Layers per stack (N)'); axes[1].set_ylabel('Parameters (millions)')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.show()
print(f'At N=6 (original Transformer base): encoder-decoder ~= {enc_dec_total[3]/1e6:.0f}M attention+FFN params,')
print(f'roughly {enc_dec_total[3]/enc_only_total[3]:.1f}x a same-depth encoder-only model — the cost of the decoder + cross-attention.')

## Common Mistakes Students Make

**Mistake 1: Thinking cross-attention is the same as self-attention.**
In self-attention, Q, K, V all come from the *same* sequence. In **cross-attention**, the Query comes from the **decoder** while Key and Value come from the **encoder**. That asymmetry is the entire point — it is how source information reaches the target.

**Mistake 2: Forgetting the causal mask in the decoder.**
Without the mask, the decoder's self-attention can see future target tokens, so during training it 'cheats' by copying the answer. The result trains beautifully and fails completely at generation. The mask must zero out the upper triangle *before* softmax.

**Mistake 3: Believing the output length must match the input length.**
The output length follows the **decoder**, not the encoder. A 5-token source can produce a 4-token (or 50-token) target. Encoder and decoder sequence lengths are independent — which is exactly why this architecture suits translation and summarization.

## Part 5: Interview Corner

**Question: Walk me through a single forward pass of a Transformer encoder-decoder, naming where each kind of attention happens.**

A complete answer:

**Encoder side.** The source tokens are embedded, given positional encodings, and passed through N encoder layers. Each layer runs **multi-head self-attention** (every source token attends to every other source token) followed by a position-wise **feed-forward network**, with **Add & Norm** around both. The final layer emits `enc` — a contextual representation of the whole source.

**Decoder side.** The target-so-far is embedded and positionally encoded, then passed through N decoder layers. Each layer runs three sub-layers: (1) **masked self-attention**, where the causal mask stops a token from seeing future tokens; (2) **cross-attention**, where the decoder forms Queries and attends to the encoder's Keys and Values — the single bridge between source and target; and (3) a **feed-forward network**. Again, **Add & Norm** wraps each sub-layer.

**Output.** The final decoder representation goes through a linear layer and softmax to give a probability distribution over the vocabulary for the next token. At inference this repeats autoregressively until an end token.

**The key insight:** there are exactly three kinds of attention — encoder self-attention (understanding the source), decoder masked self-attention (consistency with what's already generated), and cross-attention (the bridge). Self-attention and multi-head attention from Topics 2-3 are reused unchanged; the architecture is mostly about *how they are wired*.

## Key Takeaways

Five bullets you must remember for placement interviews:

- **The Transformer is two towers**: an encoder that understands the source and a decoder that generates the target — no recurrence, just stacked attention.
- **Encoder layer = 2 sub-layers, decoder layer = 3**: the decoder adds **cross-attention** because it has a second sequence (the encoder output) to consult.
- **There are exactly three kinds of attention**: encoder self-attention, decoder masked self-attention, and encoder-decoder cross-attention (Query from decoder, Key/Value from encoder).
- **Add & Norm makes depth possible**: residual connections plus LayerNorm let gradients flow through many stacked layers without vanishing.
- **Output length follows the decoder**: input and output sequences are independent in length — which is why this architecture powers translation (original Transformer), text-to-text (T5), and speech (Whisper).